# GPT-2 — Implementation / 구현

**Paper**: Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI Tech Report.

**한국어**
이 노트북은 GPT-2의 핵심 아이디어를 작은 규모로 재현합니다:
1. **causal masked self-attention** + MLP + LayerNorm으로 구성된 미니어처 GPT 디코더
2. character-level 코퍼스에서 autoregressive 학습
3. zero-shot prompting 개념 시연

GPT-2 1.5B는 48-layer × 1600-dim이지만, 이 노트북은 4-layer × 64-dim의 toy 모델로 동일한 메커니즘(pre-LN, causal mask, residual scaling)을 보여줍니다.

**English**
This notebook reproduces GPT-2's core ideas at small scale:
1. A miniature GPT decoder with **causal masked self-attention** + MLP + LayerNorm
2. Autoregressive training on a character-level corpus
3. Demonstration of the zero-shot prompting concept

GPT-2 1.5B is 48 layers × 1600 dims; this notebook uses a 4-layer × 64-dim toy model to demonstrate the same mechanisms (pre-LN, causal mask, residual scaling).

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part 1: Toy Corpus and Character Tokenizer / 토이 코퍼스와 문자 토크나이저

**한국어**
GPT-2는 byte-level BPE로 50,257개 어휘를 사용하지만, 여기서는 단순화를 위해 **character-level** 토크나이저를 사용합니다. 코퍼스는 Shakespeare 스타일의 작은 문자열로, autoregressive 학습의 모든 메커니즘을 보여주기에 충분합니다.

**English**
GPT-2 uses byte-level BPE with a vocabulary of 50,257. For simplicity, we use a **character-level** tokenizer here. The corpus is a tiny Shakespeare-style string — sufficient to demonstrate all mechanisms of autoregressive training.

In [ ]:
corpus = (
    "To be, or not to be, that is the question:\n"
    "Whether 'tis nobler in the mind to suffer\n"
    "The slings and arrows of outrageous fortune,\n"
    "Or to take arms against a sea of troubles,\n"
    "And by opposing end them. To die, to sleep,\n"
    "No more; and by a sleep to say we end\n"
    "The heart-ache and the thousand natural shocks\n"
    "That flesh is heir to: 'tis a consummation\n"
    "Devoutly to be wish'd. To die, to sleep;\n"
    "To sleep, perchance to dream - ay, there's the rub:\n"
    "For in that sleep of death what dreams may come,\n"
    "When we have shuffled off this mortal coil.\n"
) * 8  # repeat the passage to give the model more training tokens

# Build character-level vocabulary / 문자 단위 어휘 구축
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list:
    """Encode a string into a list of token IDs."""
    return [stoi[c] for c in s if c in stoi]

def decode(ids) -> str:
    """Decode a list of token IDs into a string."""
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(corpus), dtype=torch.long)
print(f'Vocab size: {vocab_size}')
print(f'Total tokens: {len(data)}')
print(f'First 60 chars: {decode(data[:60].tolist())!r}')

## Part 2: Causal Self-Attention / 인과적 셀프 어텐션

**한국어**
GPT-2의 핵심은 **causal mask**가 적용된 multi-head self-attention입니다. 토큰 $t$는 자신과 이전 토큰만 참조하며, 미래 토큰의 attention score는 $-\infty$로 설정됩니다:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

여기서 $M$은 상삼각이 $-\infty$인 mask입니다. 이로 인해 모델은 $p(x_t \mid x_{<t})$만 학습하게 됩니다.

**English**
GPT-2's heart is multi-head self-attention with a **causal mask**. Token $t$ attends only to itself and prior tokens — future-token scores are set to $-\infty$:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V$$

where $M$ is upper-triangular at $-\infty$. This forces the model to learn $p(x_t \mid x_{<t})$.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention block.

    Implements GPT-2 style scaled dot-product attention with a causal
    upper-triangular mask so each position attends only to itself and
    previous positions.
    """

    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        # Combined Q, K, V projection (efficiency)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        # Causal mask buffer; shape (1, 1, T, T)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer('causal_mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: input tensor of shape (B, T, d_model).

        Returns:
            Tensor of shape (B, T, d_model) — attention output.
        """
        B, T, C = x.shape
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.split(self.d_model, dim=-1)
        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        # Scaled dot-product
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)
        out = attn @ v  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    """Two-layer feed-forward MLP with GELU activation (GPT-2 style)."""

    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    """Single Transformer decoder block.

    Note the GPT-2 modification: LayerNorm is applied to the **input** of
    each sub-block (pre-LN), then residual is added to the unnormalized
    input — improving stability of deep stacks.
    """

    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN, residual connection
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


print('Building blocks defined: CausalSelfAttention, MLP, Block')

## Part 3: Tiny GPT Model / 작은 GPT 모델

**한국어**
전체 GPT 모델은: token embedding + positional embedding → $N$개의 Block → 최종 LayerNorm → vocabulary 분포로의 projection. GPT-2는 token embedding과 output projection을 **weight tying**합니다.

또한 GPT-2의 $1/\sqrt{N}$ residual scaling을 따라, residual layer 가중치를 추가로 스케일링합니다.

**English**
Full GPT model: token embedding + positional embedding → $N$ Blocks → final LayerNorm → projection to vocabulary distribution. GPT-2 **weight-ties** the token embedding and the output projection.

We also follow GPT-2's $1/\sqrt{N}$ residual scaling — re-initializing residual-layer weights with this factor.

In [ ]:
@dataclass
class GPTConfig:
    """Configuration for a tiny GPT model."""

    vocab_size: int
    block_size: int = 64
    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 64
    dropout: float = 0.1


class TinyGPT(nn.Module):
    """A tiny GPT-2-style autoregressive language model.

    Implements the same architectural choices as GPT-2 at miniature scale:
    pre-LayerNorm, learned positional embeddings, causal masking, and
    1/sqrt(N) residual scaling at initialization.
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [Block(cfg.d_model, cfg.n_heads, cfg.block_size, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # Weight tying (GPT-2 trick)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)
        # Residual init scaling: 1/sqrt(N) where N = number of residual layers
        n_residual = 2 * cfg.n_layers  # attn + MLP per block
        for name, p in self.named_parameters():
            if name.endswith('proj.weight') or name.endswith('fc2.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_residual))

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        """Initialize weights GPT-2 style: N(0, 0.02) for linear/embedding."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        """Forward pass.

        Args:
            idx: token IDs of shape (B, T).
            targets: optional target IDs of shape (B, T) for cross-entropy.

        Returns:
            (logits, loss) — loss is None when targets is None.
        """
        B, T = idx.shape
        assert T <= self.cfg.block_size, 'sequence too long'
        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None):
        """Autoregressive generation loop.

        At each step, condition on the most recent ``block_size`` tokens, sample
        the next token from the resulting distribution, and append it.
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


cfg = GPTConfig(vocab_size=vocab_size, block_size=64, n_layers=4, n_heads=4, d_model=64)
model = TinyGPT(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Tiny GPT parameters: {n_params:,}')

## Part 4: Training Loop / 학습 루프

**한국어**
GPT-2는 standard cross-entropy NLL을 최소화합니다. 각 batch는 corpus에서 random offset을 샘플링한 (input, target) 쌍이며, target은 input을 한 칸 shift한 것입니다 — 이로써 next-token 예측을 학습합니다.

**English**
GPT-2 minimizes standard cross-entropy NLL. Each batch is a (input, target) pair sampled at random offsets from the corpus, with target shifted one position — training next-token prediction.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, block_size: int):
    """Sample a random batch of (input, target) pairs from a 1D token tensor."""
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
n_steps = 600
batch_size = 32

model.train()
loss_history = []
for step in range(n_steps):
    xb, yb = get_batch(data, batch_size, cfg.block_size)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if step % 100 == 0 or step == n_steps - 1:
        ppl = math.exp(loss.item())
        print(f'step {step:4d}  loss {loss.item():.4f}  ppl {ppl:.2f}')

print(f'\nFinal loss: {loss_history[-1]:.4f}  (perplexity {math.exp(loss_history[-1]):.2f})')

## Part 5: Autoregressive Generation / 자기회귀 생성

**한국어**
학습된 모델로 샘플을 생성합니다. 짧은 prompt에서 시작해 한 토큰씩 autoregressive하게 확장합니다. temperature=1.0, top-$k$=10으로 다양성 vs. coherence 균형.

**English**
We sample from the trained model — starting from a short prompt and extending one token at a time, autoregressively. We use temperature=1.0 and top-$k$=10 for a balance of diversity and coherence.

In [ ]:
model.eval()
prompt = 'To be, or'
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = model.generate(prompt_ids, max_new_tokens=120, temperature=1.0, top_k=10)
print('=' * 50)
print('Generated continuation / 생성된 텍스트:')
print('=' * 50)
print(decode(out[0].tolist()))

## Part 6: Zero-Shot Prompting Concept / 제로샷 프롬프팅 개념

**한국어**
GPT-2의 핵심 통찰: **task를 자연어로 표현하면, fine-tuning 없이 LM이 task를 수행할 수 있다.**

우리의 toy 모델은 작은 corpus로 학습됐기 때문에 의미 있는 zero-shot은 불가능하지만, **메커니즘**은 시연할 수 있습니다. 두 prompt를 비교:
1. `"To be, or"` — corpus의 자연 시작
2. `"\nTL;DR: "` — GPT-2가 요약을 유도하는 데 쓴 prompt

두 prompt의 다음 토큰 확률 분포를 비교하면, prompt context가 LM 출력을 어떻게 변화시키는지 직접 볼 수 있습니다. 이는 GPT-2 1.5B에서 ROUGE 6.4가 "TL;DR" 추가만으로 달라진 것과 같은 메커니즘입니다.

**English**
GPT-2's core insight: **express the task in natural language, and the LM can perform it without fine-tuning.**

Our toy model is too small for meaningful zero-shot, but the **mechanism** can still be demonstrated. Compare two prompts:
1. `"To be, or"` — a natural corpus opening
2. `"\nTL;DR: "` — the prompt GPT-2 used to induce summarization

Looking at the next-token distributions reveals how prompt context shifts the LM's output — the same mechanism that gave GPT-2 1.5B a 6.4-ROUGE jump from adding "TL;DR:".

In [ ]:
import numpy as np

@torch.no_grad()
def next_token_distribution(model: TinyGPT, prompt: str, top_k: int = 5):
    """Compute and print the top-k next-token probabilities for a prompt."""
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    logits, _ = model(ids)
    probs = F.softmax(logits[0, -1], dim=-1).cpu().numpy()
    top = np.argsort(-probs)[:top_k]
    print(f'Prompt: {prompt!r}')
    for i in top:
        ch = itos[int(i)]
        ch_repr = repr(ch) if ch in ('\n', ' ', '\t') else ch
        print(f'  {ch_repr:6s}  prob={probs[i]:.4f}')
    print()


# Compare two prompts to see how context steers the distribution
next_token_distribution(model, 'To be, or')
next_token_distribution(model, 'and arrows of')
next_token_distribution(model, 'sleep, perchance')

## Summary / 요약

**한국어**
이 노트북에서 GPT-2의 핵심 메커니즘을 모두 구현했습니다: causal self-attention, pre-LN block, residual scaling, weight-tied embedding, autoregressive generation. 작은 toy corpus에서도 모델이 next-token 분포를 학습함을 확인했습니다. 실제 GPT-2의 zero-shot 능력은 같은 메커니즘 + (1) WebText의 다양성, (2) 1.5B 파라미터로 emergent하게 나타납니다.

**English**
This notebook implements all of GPT-2's core mechanisms: causal self-attention, pre-LN blocks, residual scaling, weight-tied embeddings, and autoregressive generation. Even on a tiny toy corpus, the model learns the next-token distribution. GPT-2's actual zero-shot capability emerges from the same mechanisms plus (1) WebText diversity and (2) 1.5B parameters.

| Concept / 개념 | This Notebook / 이 노트북 | GPT-2 Paper / 원논문 |
|---|---|---|
| Tokenization | character-level | byte-level BPE, vocab 50,257 |
| Layers | 4 | 48 |
| $d_{\text{model}}$ | 64 | 1600 |
| Parameters | ~50K | 1.542B |
| Context size | 64 | 1024 |
| Training corpus | tiny Shakespeare | WebText (40GB) |
| LayerNorm position | pre-LN (input of sub-block) | pre-LN + final LN |
| Residual init scaling | $1/\sqrt{2N}$ | $1/\sqrt{N}$ |
| Generation | top-$k$ sampling | top-$k$, nucleus, greedy |
| Evaluation | qualitative samples | zero-shot on 8+ benchmarks |